# 12 - Market Regime Detection

**Project:** AI Quantitative Trading & Strategy Optimisation System  
**Phase:** 6 — Feature Engineering and Market Regime Detection  
**Status:** Executed against the recorded local acquisitions on 14 November 2026; owner review is pending.

## 1. Objective and scope

Investigate interpretable trend and volatility states for daily market series. Regime indicators are descriptive features, not permanent economic labels, forecast targets, or trading signals. The calculations are asset-agnostic and use each instrument’s own `Close` series.

This notebook does not train a forecasting model, develop a strategy, or infer causal economic regimes. It begins with transparent rolling indicators. GMM and HMM are not used: the current research question can be addressed with interpretable past-only rules, and added model complexity is not yet justified.

## 2. Candidate methodologies

### 2.1 Relative-volatility state

For a candidate 20-session window, realised volatility is the square root of the sum of squared daily log returns in that window. The low and high cut-offs are the 33rd and 67th percentiles of the preceding 252 completed realised-volatility observations. The threshold series is shifted by one observation, so the current volatility value cannot determine its own cut-offs. Values at or below the lower cut-off are labelled `low_volatility`; values above the upper cut-off are `high_volatility`; values between them are `typical_volatility`.

The 20-session window, 252-observation history and 33/67 percentiles are starting research conventions, not optimised or approved permanent parameters. Sensitivity to alternative quantiles and estimation windows must be reviewed. A rolling historical distribution adapts to changing conditions but can also make labels relative to a changing baseline.

### 2.2 Trend state

For candidate 50- and 200-session simple moving averages, `uptrend` requires Close > SMA50 > SMA200; `downtrend` requires Close < SMA50 < SMA200; all other observations with both averages available are `mixed_trend`. This rule uses no future observations and has no fitted threshold. Its long warm-up period and sensitivity to the chosen windows limit its early coverage and stability.

### 2.3 Information timing

Indicators indexed by session *t* include that session’s Close and are available only after its close. Rolling quantile cut-offs at *t* use realised-volatility observations only through *t−1*. Any later decision must occur after the information is available and simulated execution must be no earlier than the next eligible session. No same-bar execution is assumed.

## 3. Data and temporal limitations

The notebook reads the configured market acquisition files without modifying them. Close-to-close price returns exclude dividends; corporate-action and source-revision limitations remain. The price-unit convention for LSE equities is not historically validated, so no conversion is performed. Trend relationships and return volatility are scale invariant to a constant quote-unit conversion.

Missing observations are not filled. Windows count rows in the source series, which are observed market sessions but have not been reindexed against a separately validated exchange calendar here. A missing close interrupts returns and causes affected rolling windows to remain missing. Same-date daily observations do not establish precise vendor publication timestamps or strict historical PIT eligibility.

## 4. Method and implementation

The reusable function in `src/regimes.py` returns realised volatility, lagged historical cut-offs, volatility labels, two moving averages and trend labels. It validates chronological unique indexes and positive finite prices, preserves missing values, and requires complete rolling windows. No full-sample quantile, clustering, HMM, GMM or future-informed classification is used.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import yaml

PROJECT_ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "configs" / "assets").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.regimes import build_regime_indicators

ASSET_CONFIG_PATHS = sorted((PROJECT_ROOT / "configs" / "assets").glob("*.yaml"))
assert ASSET_CONFIG_PATHS, "No configured market assets were found."
print(f"Configured assets: {len(ASSET_CONFIG_PATHS)}")

Configured assets: 4


## 5. Load configured daily closing prices

Asset symbols come from configuration. The notebook requires exactly one acquired Parquet file for each symbol and stops if the source is absent or ambiguous. The read is non-mutating.

In [2]:
market_data = {}
source_inventory = []
for config_path in ASSET_CONFIG_PATHS:
    with config_path.open(encoding="utf-8") as config_file:
        config = yaml.safe_load(config_file)
    symbol = config["provider"]["symbol"]
    candidates = sorted(
        (PROJECT_ROOT / "data" / "raw" / "market" / "yahoo_finance" / symbol)
        .glob(f"acquisition_*/{symbol}.parquet")
    )
    if len(candidates) != 1:
        raise RuntimeError(f"Expected exactly one acquired Parquet file for {symbol}; found {len(candidates)}.")

    frame = pd.read_parquet(candidates[0])
    if "Date" in frame.columns:
        frame = frame.set_index("Date")
    frame.index = pd.to_datetime(frame.index)
    frame.index.name = "observation_date"
    if not frame.index.is_monotonic_increasing or frame.index.has_duplicates:
        raise ValueError(f"{symbol} dates must be unique and chronological.")
    if "Close" not in frame.columns:
        raise ValueError(f"No Close field is available for {symbol}.")

    market_data[symbol] = frame
    source_inventory.append({
        "symbol": symbol,
        "instrument_type": config["asset"]["instrument_type"],
        "price_unit": config["asset"]["price_unit"],
        "price_unit_validation_status": config["asset"]["price_unit_validation_status"],
        "source_file": candidates[0].relative_to(PROJECT_ROOT).as_posix(),
        "observations": len(frame),
        "first_observation": frame.index.min(),
        "last_observation": frame.index.max(),
    })

source_inventory = pd.DataFrame(source_inventory).set_index("symbol")
source_inventory

,instrument_type,price_unit,price_unit_validation_status,source_file,observations,first_observation,last_observation
symbol,,,,,,,
^FTSE,index,index_points,not_applicable,data/raw/market/yahoo_finance/^FTSE/acquisitio...,11004,1984-01-03,2026-11-02
NG.L,equity,GBp,pending,data/raw/market/yahoo_finance/NG.L/acquisition...,7890,1995-12-11,2026-11-02
SSE.L,equity,GBp,pending,data/raw/market/yahoo_finance/SSE.L/acquisitio...,9060,1991-06-17,2026-11-02
SVT.L,equity,GBp,pending,data/raw/market/yahoo_finance/SVT.L/acquisitio...,9455,1989-12-11,2026-11-02


## 6. Construct baseline regime indicators

The baseline parameters are explicit and can be varied for sensitivity analysis. No parameters are selected by maximising returns or fitting to future observations.

In [3]:
BASELINE = {
    "volatility_window": 20,
    "threshold_window": 252,
    "volatility_quantiles": (0.33, 0.67),
    "trend_windows": (50, 200),
}

regime_indicators = {
    symbol: build_regime_indicators(frame, **BASELINE)
    for symbol, frame in market_data.items()
}

regime_coverage = pd.DataFrame([
    {
        "symbol": symbol,
        "volatility_labels": int(frame["volatility_regime"].notna().sum()),
        "trend_labels": int(frame["trend_regime"].notna().sum()),
        "low_volatility_share": frame["volatility_regime"].eq("low_volatility").mean(),
        "typical_volatility_share": frame["volatility_regime"].eq("typical_volatility").mean(),
        "high_volatility_share": frame["volatility_regime"].eq("high_volatility").mean(),
        "uptrend_share": frame["trend_regime"].eq("uptrend").mean(),
        "mixed_trend_share": frame["trend_regime"].eq("mixed_trend").mean(),
        "downtrend_share": frame["trend_regime"].eq("downtrend").mean(),
    }
    for symbol, frame in regime_indicators.items()
]).set_index("symbol")
regime_coverage

,volatility_labels,trend_labels,low_volatility_share,typical_volatility_share,high_volatility_share,uptrend_share,mixed_trend_share,downtrend_share
symbol,,,,,,,,
^FTSE,4449,4668,0.365925,0.293100,0.340976,0.443445,0.398458,0.158098
NG.L,7617,7690,0.351188,0.284364,0.364448,0.384265,0.423667,0.192068
SSE.L,8787,8860,0.328554,0.310686,0.360760,0.392099,0.407336,0.200564
SVT.L,9182,9255,0.351666,0.292420,0.355914,0.391896,0.441167,0.166937


In [4]:
# Inspect the latest rows for one configured asset; labels remain descriptive.
example_symbol = next(iter(regime_indicators))
regime_indicators[example_symbol].tail(10)

,realised_volatility_20,volatility_threshold_low,volatility_threshold_high,volatility_regime,sma_50,sma_200,trend_regime
observation_date,,,,,,,
2026-10-20,0.030652,0.025512,0.031665,typical_volatility,10711.443965,10499.694492,uptrend
2026-10-21,0.030707,0.025750,0.031665,typical_volatility,10709.617969,10503.217993,uptrend
2026-10-22,0.030726,0.025946,0.031665,typical_volatility,10707.453965,10506.619492,uptrend
2026-10-23,0.030862,0.026063,0.031665,typical_volatility,10705.807969,10509.448496,mixed_trend
2026-10-26,0.031668,0.026128,0.031665,high_volatility,10706.151973,10512.581494,uptrend
2026-10-27,0.032236,0.026170,0.031666,high_volatility,10705.475977,10515.326992,mixed_trend
2026-10-28,0.032115,0.026264,0.031678,high_volatility,10704.805977,10517.877490,mixed_trend
2026-10-29,0.027432,0.026538,0.031733,typical_volatility,10704.379961,10520.293486,uptrend
2026-10-30,0.027251,0.026625,0.031733,typical_volatility,10704.011953,10522.765986,uptrend


## 7. Sensitivity review

Parameter sensitivity is part of the research question. The table below compares candidate rolling-history lengths and quantile pairs without selecting a preferred result. Once executed, compare label coverage, transitions and historical stability; do not choose a configuration because its chart appears attractive or because it aligns with a desired outcome.

In [5]:
SENSITIVITY_CANDIDATES = [
    {"threshold_window": 126, "volatility_quantiles": (0.20, 0.80)},
    {"threshold_window": 252, "volatility_quantiles": (0.33, 0.67)},
    {"threshold_window": 504, "volatility_quantiles": (0.20, 0.80)},
]

sensitivity_rows = []
for symbol, source in market_data.items():
    for candidate in SENSITIVITY_CANDIDATES:
        parameters = {**BASELINE, **candidate}
        result = build_regime_indicators(source, **parameters)
        labels = result["volatility_regime"]
        sensitivity_rows.append({
            "symbol": symbol,
            "threshold_window": candidate["threshold_window"],
            "quantiles": candidate["volatility_quantiles"],
            "classified_observations": int(labels.notna().sum()),
            "low_share": labels.eq("low_volatility").mean(),
            "typical_share": labels.eq("typical_volatility").mean(),
            "high_share": labels.eq("high_volatility").mean(),
        })

sensitivity_summary = pd.DataFrame(sensitivity_rows).set_index(
    ["symbol", "threshold_window", "quantiles"]
)
sensitivity_summary

classified_observations  low_share  \
symbol threshold_window quantiles                                          
^FTSE  126              (0.2, 0.8)                       4827   0.249223   
       252              (0.33, 0.67)                     4449   0.365925   
       504              (0.2, 0.8)                       3693   0.256702   
NG.L   126              (0.2, 0.8)                       7743   0.235826   
       252              (0.33, 0.67)                     7617   0.351188   
       504              (0.2, 0.8)                       7365   0.220910   
SSE.L  126              (0.2, 0.8)                       8913   0.234826   
       252              (0.33, 0.67)                     8787   0.328554   
       504              (0.2, 0.8)                       8535   0.217809   
SVT.L  126              (0.2, 0.8)                       9308   0.229480   
       252              (0.33, 0.67)                     9182   0.351666   
       504              (0.2, 0.8)                       8930   0.219261   

                                      typical_share  high_share  
symbol threshold_window quantiles                                
^FTSE  126              (0.2, 0.8)         0.546924    0.203853  
       252              (0.33, 0.67)       0.293100    0.340976  
       504              (0.2, 0.8)         0.509071    0.234227  
NG.L   126              (0.2, 0.8)         0.534676    0.229498  
       252              (0.33, 0.67)       0.284364    0.364448  
       504              (0.2, 0.8)         0.552206    0.226884  
SSE.L  126              (0.2, 0.8)         0.536744    0.228430  
       252              (0.33, 0.67)       0.310686    0.360760  
       504              (0.2, 0.8)         0.544815    0.237376  
SVT.L  126              (0.2, 0.8)         0.527073    0.243446  
       252              (0.33, 0.67)       0.292420    0.355914  
       504              (0.2, 0.8)         0.545241    0.235498

## 8. Transition and sub-period diagnostics

Regime proportions alone can hide frequent switching or historical instability. The following summaries count transitions between adjacent non-missing source rows and show volatility-state proportions by decade. Adjacent rows are not necessarily guaranteed to be consecutive exchange sessions where the source has gaps; these remain descriptive diagnostics.

In [6]:
transition_rows = []
decade_rows = []
for symbol, indicators in regime_indicators.items():
    for state_column in ("volatility_regime", "trend_regime"):
        state = indicators[state_column]
        previous = state.shift(1)
        comparable = state.notna() & previous.notna()
        pairs = int(comparable.sum())
        changes = int((state.loc[comparable] != previous.loc[comparable]).sum())
        transition_rows.append({
            "symbol": symbol,
            "state_type": state_column,
            "adjacent_label_pairs": pairs,
            "transitions": changes,
            "transition_rate": changes / pairs if pairs else float("nan"),
        })
    volatility_state = indicators["volatility_regime"]
    decade = (volatility_state.index.year // 10) * 10
    for decade_start, labels in volatility_state.groupby(decade):
        available = labels.dropna()
        if available.empty:
            continue
        shares = available.value_counts(normalize=True)
        decade_rows.append({
            "symbol": symbol,
            "decade": int(decade_start),
            "classified_observations": len(available),
            "low_share": float(shares.get("low_volatility", 0.0)),
            "typical_share": float(shares.get("typical_volatility", 0.0)),
            "high_share": float(shares.get("high_volatility", 0.0)),
        })

transition_summary = pd.DataFrame(transition_rows).set_index(["symbol", "state_type"])
decade_volatility_summary = pd.DataFrame(decade_rows).set_index(["symbol", "decade"])
transition_summary, decade_volatility_summary

(                          adjacent_label_pairs  transitions  transition_rate
 symbol state_type                                                           
 ^FTSE  volatility_regime                  4446          410         0.092218
        trend_regime                       4665          429         0.091961
 NG.L   volatility_regime                  7616          726         0.095326
        trend_regime                       7689          711         0.092470
 SSE.L  volatility_regime                  8786          829         0.094355
        trend_regime                       8859          739         0.083418
 SVT.L  volatility_regime                  9181          938         0.102168
        trend_regime                       9254          812         0.087746,
                classified_observations  low_share  typical_share  high_share
 symbol decade                                                               
 ^FTSE  2000                        743   0.294751       0.2584

## 9. Interpretation framework and limitations

- Volatility labels are relative to the preceding estimation window, not fixed absolute risk bands. They may change when the historical distribution shifts.
- Trend labels are rule-based summaries of price and moving-average ordering; `mixed_trend` combines several distinct configurations.
- The 50/200-session trend rule has a long warm-up and may respond slowly to turning points.
- The executed transition and decade summaries show that labels switch over time and state proportions vary by sub-period. Rolling indicators can remain sensitive to missing observations, window choices and structural breaks. Adjacent regime labels are not independent evidence.
- These labels are not proof of stable or permanent market regimes and do not imply that a regime-specific model or strategy would help.
- The market source has historical quality, licensing and revision limitations. No macroeconomic variables enter the classifiers.
- No predictive model, trading rule, backtest or same-session execution assumption is introduced.

### Evidence status

- **Implemented in this change:** causal rolling volatility thresholds and moving-average trend states in `src/regimes.py`; deterministic edge and temporal tests.
- **Executed on 14 November 2026:** data loading, baseline regimes, parameter sensitivity, transition rates and decade summaries; outputs are stored here. Targeted regime tests passed.
- **Reused:** earlier Phase 3 temporal and market-data findings and Phase 5 volatility/autocorrelation evidence. Those results motivate investigation but do not validate these state definitions.
- **Not validated:** stable economic regimes, predictive usefulness, parameter optimality, or strict PIT status of the historical market series.
